0) Paths, env, folders

In [16]:
# --- config paths ---
PROJ = "/Users/kaychen/Documents/projects/cgm_insights/"
CGMF = "/Users/kaychen/Documents/projects/CGMformer/"
CKPT = f"{CGMF}/cgm_ckp/checkpoint-30000"

import os, sys, pathlib, warnings
warnings.filterwarnings("ignore")

sys.path.append(CGMF)

os.makedirs(f"{PROJ}/data/embeddings/", exist_ok=True)
os.makedirs(f"{PROJ}/data/models/cgmformer_diet", exist_ok=True)

# device selection (CUDA -> MPS -> CPU)
import torch
if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
DEVICE

'mps'

# 1. Load processed data

In [28]:
import pandas as pd

# Your 5‑minute glucose time series
g5 = pd.read_csv(f"{PROJ}data/processed/processed_glucose_5min.csv",
                 parse_dates=["start"])
g5 = g5.set_index("start").sort_index()
assert "glucose_mgdl" in g5.columns, g5.columns

# Your meals dataset (the one you built for the baseline)
# If you saved it with another name, update this path.
ds = pd.read_csv(f"{PROJ}data/processed/meals_with_targets.csv",
                 parse_dates=["meal_time"])
ds = ds.sort_values("meal_time").reset_index(drop=True)

# quick peek
g5.head(), ds.head()


(                           glucose_mgdl
 start                                  
 2025-07-28 23:00:00+00:00      74.93508
 2025-07-28 23:05:00+00:00      83.92734
 2025-07-28 23:10:00+00:00      88.92306
 2025-07-28 23:15:00+00:00      95.91696
 2025-07-28 23:20:00+00:00      88.92306,
    meal_id  carbohydrates  energyconsumed  fattotal     fiber   protein  \
 0        1       95.36570        1076.849   46.8651  15.25000  75.49040   
 1        2       39.69750         408.376   16.8392   4.27695  27.50140   
 2        3       63.92252         919.279   49.4770  12.62195  58.46362   
 3        4       36.00670         359.890   14.9160   6.75300  23.78660   
 4        5        3.87900         104.220    8.9874   2.25000   3.80700   
 
       sugar                 meal_time     carbs      fat  ...  \
 0  14.08500 2025-07-30 12:00:00+00:00  95.36570  46.8651  ...   
 1  15.30210 2025-07-30 13:37:00+00:00  39.69750  16.8392  ...   
 2  25.40512 2025-07-31 11:52:00+00:00  63.92252  49.477

# 2. Build pre‑meal CGM sequences and discretize to tokens

In [29]:
import numpy as np
from datetime import timedelta

T = 36                    # 3h @ 5‑min
STEP = pd.Timedelta(minutes=5)

def extract_premeal_seq(g5_df, t, T=36, step=STEP, col="glucose_mgdl"):
    """
    Returns a length‑T array of mg/dL ending at time t (exclusive),
    sampled every 5 minutes. Missing -> NaN.
    """
    idx = pd.date_range(end=t-step, periods=T, freq=step)
    s = g5_df[col].reindex(idx)
    return s.values.astype(float)

def discretize_to_tokens(arr):
    """
    Clip to [39, 301], round to int. If NaN, fill with row median (fallback=100).
    """
    x = arr.copy()
    if np.all(np.isnan(x)):
        x[:] = 100.0
    else:
        med = np.nanmedian(x)
        if np.isnan(med): med = 100.0
        x = np.where(np.isnan(x), med, x)
    x = np.clip(np.rint(x), 39, 301).astype(int)
    return x

# Build [N, T] token matrix for all meals
tokens = np.vstack([discretize_to_tokens(extract_premeal_seq(g5, t, T=T))
                    for t in ds["meal_time"]])
tokens.shape


(49, 36)

# 3. Load tokenizer maps + pretrained CGMformer

In [30]:
import os, sys, pickle, inspect, importlib
import torch

CGMF = "/Users/kaychen/Documents/projects/CGMformer"  # repo root
CKP_DIR = f"{CGMF}/cgm_ckp/checkpoint-30000"           # <- your screenshot
TOK2ID =  f"{CGMF}/cgm_ckp/token2id.pkl"
ID2TOK =  f"{CGMF}/cgm_ckp/id2token.pkl"

sys.path.append(CGMF)

# --- load vocab maps (useful if the repo expects string tokens) ---
with open(TOK2ID, "rb") as f:
    token2id = pickle.load(f)
with open(ID2TOK, "rb") as f:
    id2token = pickle.load(f)

len(token2id), len(id2token)


(266, 266)

## 3a. Model Loading

In [31]:
# Try HF AutoModel path first
hf_loaded = False
try:
    from transformers import AutoConfig, AutoModel
    cfg = AutoConfig.from_pretrained(CKP_DIR)
    base_model = AutoModel.from_pretrained(CKP_DIR, config=cfg)
    hf_loaded = True
    print("Loaded via HuggingFace AutoModel. Hidden size:", base_model.config.hidden_size)
except Exception as e:
    print("HF load failed -> will try repo class:\n", e)
    hf_loaded = False

if not hf_loaded:
    cm = importlib.import_module("CGMFormer.model")  # file: CGMformer/CGMFormer/model.py
    # inspect available classes
    print([n for n, obj in cm.__dict__.items() if inspect.isclass(obj)])
    # === pick the correct class name from the printed list ===
    # Example (you might need to adjust the class & init args):
    # from CGMFormer.model import CGMFormer
    # base_model = CGMFormer.from_pretrained(CKP_DIR)

# Put on device & eval
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
base_model = base_model.to(device).eval()
device


Some weights of BertModel were not initialized from the model checkpoint at /Users/kaychen/Documents/projects/CGMformer/cgm_ckp/checkpoint-30000 and are newly initialized: ['bert.embeddings.position_embeddings.weight', 'bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded via HuggingFace AutoModel. Hidden size: 128


'mps'

# Convert interger tokens (39-391) to model IDs

In [32]:
import numpy as np

def mgdl_tokens_to_model_ids(tokens_2d, token2id):
    """
    tokens_2d: np.ndarray [N, T] of ints (mg/dL rounded & clipped 39..301)
    Returns: np.ndarray [N, T] of ids according to token2id (if keys exist).
    If string key not in vocab, leaves value as-is (identity).
    """
    ids = np.empty_like(tokens_2d)
    missed = 0
    for i in range(tokens_2d.shape[0]):
        for j in range(tokens_2d.shape[1]):
            s = str(int(tokens_2d[i,j]))
            if s in token2id:
                ids[i,j] = token2id[s]
            else:
                # fall back: maybe the model was trained with numeric ids directly
                ids[i,j] = tokens_2d[i,j]
                missed += 1
    if missed:
        print(f"Note: {missed} tokens were not in token2id; left as-is.")
    return ids

# `tokens` comes from Step 2 (shape [N,T])
model_input_ids = mgdl_tokens_to_model_ids(tokens, token2id)
model_input_ids.shape, model_input_ids[:2, :6]


Note: 1764 tokens were not in token2id; left as-is.


((49, 36),
 array([[ 74,  71,  72,  71,  71,  73],
        [100, 100, 100, 100, 100, 100]]))

# 5. Embed sequences with CGMformer

In [33]:
import torch
from torch.utils.data import TensorDataset, DataLoader

input_ids = torch.tensor(model_input_ids, dtype=torch.long)
attn_mask = torch.ones_like(input_ids)

loader = DataLoader(TensorDataset(input_ids, attn_mask), batch_size=128, shuffle=False)

all_embeds = []
with torch.no_grad():
    for ids, mask in loader:
        ids  = ids.to(device)
        mask = mask.to(device)
        out  = base_model(input_ids=ids, attention_mask=mask)

        if hasattr(out, "last_hidden_state"):
            hidden = out.last_hidden_state        # [B,T,H]
            emb = hidden[:, 0]                    # CLS
        elif isinstance(out, (list, tuple)):
            hidden = out[0]
            emb = hidden[:, 0]
        else:
            emb = out                              # already pooled

        all_embeds.append(emb.detach().cpu())

embeddings = torch.cat(all_embeds, dim=0).numpy()
embeddings.shape


(49, 128)

In [34]:
import numpy as np, os
EMB_PATH = f"/Users/kaychen/Documents/projects/cgm_insights/data/embeddings/cgmformer_premeal_T{tokens.shape[1]}.npy"
np.save(EMB_PATH, embeddings)
EMB_PATH


'/Users/kaychen/Documents/projects/cgm_insights/data/embeddings/cgmformer_premeal_T36.npy'

In [35]:
ds.columns

Index(['meal_id', 'carbohydrates', 'energyconsumed', 'fattotal', 'fiber',
       'protein', 'sugar', 'meal_time', 'carbs', 'fat', 'kcal', 'net_carbs',
       'hour', 'dow', 'hr_mean_30m', 'active_kcal_60m', 'steps_60m',
       'walk_run_dist_60m', 'rest_hr_daily', 'exercise_min_60m', 'flights_60m',
       'base', 'peak', 'd_peak', 'auc_0_2h', 'carbs_per_100kcal',
       'fat_per_100kcal', 'protein_per_100kcal', 'sugar_per_100kcal',
       'fiber_per_100kcal', 'carb_to_fat', 'carb_to_protein', 'fiber_to_carb',
       'sugar_frac_carb', 'prev_meal_kcal', 'prev_meal_carbs', 'prev_meal_fat',
       'prev_meal_protein', 'prev_meal_fiber'],
      dtype='object')

# 6. Fuse embeddings + meal features

In [36]:
def time_split_3way(df, val_frac=0.1, test_frac=0.2, time_col="meal_time"):
    df = df.sort_values(time_col).reset_index(drop=True)
    n = len(df)
    n_test = int(n * test_frac)
    n_val  = int(n * val_frac)
    n_train = n - n_val - n_test
    idx_tr = np.arange(0, n_train)
    idx_va = np.arange(n_train, n_train + n_val)
    idx_te = np.arange(n_train + n_val, n)
    return idx_tr, idx_va, idx_te

idx_tr, idx_va, idx_te = time_split_3way(ds, val_frac=0.1, test_frac=0.2, time_col="meal_time")

# Tabular features
TAB_FEATURES = [
    "kcal",
    "sugar_per_100kcal",        # instead of both sugar+carb
    "prev_meal_kcal",
    "hr_mean_30m","rest_hr_daily",
    "steps_60m","dow"
]

# Build matrices for each split
X_embed = embeddings
X_tab   = ds[TAB_FEATURES].astype(float).fillna(0.0).values
y1      = ds["d_peak"].astype(float).values
y2      = ds["auc_0_2h"].astype(float).values

Xe_tr, Xe_va, Xe_te = X_embed[idx_tr], X_embed[idx_va], X_embed[idx_te]
Xt_tr, Xt_va, Xt_te = X_tab[idx_tr],   X_tab[idx_va],   X_tab[idx_te]
y1_tr, y1_va, y1_te = y1[idx_tr],      y1[idx_va],      y1[idx_te]
y2_tr, y2_va, y2_te = y2[idx_tr],      y2[idx_va],      y2[idx_te]

Xe_tr.shape, Xe_va.shape, Xe_te.shape


((36, 128), (4, 128), (9, 128))

# 7. Dataloaders + small head

In [37]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

class DietHead(nn.Module):
    def __init__(self, d_emb, d_tab, hidden=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_emb + d_tab, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 2)  # [d_peak, auc_0_2h]
        )
    def forward(self, e, x):
        return self.mlp(torch.cat([e, x], dim=1))

def make_loader(E, X, y1, y2, bs=64, shuffle=False):
    to = lambda a,dt: torch.tensor(a, dtype=dt)
    ds = TensorDataset(to(E, torch.float32), to(X, torch.float32),
                       to(y1, torch.float32), to(y2, torch.float32))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)

trL = make_loader(Xe_tr, Xt_tr, y1_tr, y2_tr, shuffle=True)
vaL = make_loader(Xe_va, Xt_va, y1_va, y2_va)
teL = make_loader(Xe_te, Xt_te, y1_te, y2_te)

head   = DietHead(d_emb=Xe_tr.shape[1], d_tab=Xt_tr.shape[1]).to(device)
opt    = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
lossfn = nn.SmoothL1Loss()


# 8. Train on train, pick best on val (early stopping on val loss)

In [38]:
def run_epoch(loader, train=True):
    head.train(train)
    total, n = 0.0, 0
    mae1 = mae2 = 0.0
    for E, X, t1, t2 in loader:
        E, X, t1, t2 = [d.to(device) for d in (E, X, t1, t2)]
        with torch.set_grad_enabled(train):
            p = head(E, X)         # [B,2]
            p1, p2 = p[:,0], p[:,1]
            loss = lossfn(p1, t1) + lossfn(p2, t2)
        if train:
            opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(E); n += len(E)
        mae1 += (p1 - t1).abs().sum().item()
        mae2 += (p2 - t2).abs().sum().item()
    return total / n, mae1 / n, mae2 / n

best_val = (float("inf"), None)
EPOCHS = 30

for ep in range(1, EPOCHS+1):
    tr = run_epoch(trL, True)
    va = run_epoch(vaL, False)
    if va[0] < best_val[0]:
        best_val = (va[0], {k: v.cpu().clone() for k, v in head.state_dict().items()})
    if ep % 5 == 0:
        print(f"ep{ep:02d}  train_loss={tr[0]:.3f}  val_loss={va[0]:.3f}  "
              f"val_MAE Δpeak={va[1]:.2f}  val_MAE AUC={va[2]:.2f}")

# restore best-by-val
head.load_state_dict(best_val[1])
torch.save(head.state_dict(), f"{PROJ}/models/cgmformer_diet/head_best.pt")


ep05  train_loss=44.211  val_loss=81.228  val_MAE Δpeak=45.09  val_MAE AUC=37.13
ep10  train_loss=37.750  val_loss=82.654  val_MAE Δpeak=41.62  val_MAE AUC=42.03
ep15  train_loss=33.612  val_loss=99.420  val_MAE Δpeak=44.62  val_MAE AUC=55.80
ep20  train_loss=32.177  val_loss=106.602  val_MAE Δpeak=53.80  val_MAE AUC=53.81
ep25  train_loss=30.643  val_loss=128.659  val_MAE Δpeak=64.40  val_MAE AUC=65.26
ep30  train_loss=29.908  val_loss=124.839  val_MAE Δpeak=59.13  val_MAE AUC=66.71


# 9. Test Metrics

In [39]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

SHOW_TEST = False   # set False to skip printing now

def eval_loader(loader):
    head.eval()
    p1s, p2s, g1s, g2s = [], [], [], []
    with torch.no_grad():
        for E, X, t1, t2 in loader:
            E, X = E.to(device), X.to(device)
            P = head(E, X)
            p1s.append(P[:,0].cpu().numpy()); p2s.append(P[:,1].cpu().numpy())
            g1s.append(t1.numpy());           g2s.append(t2.numpy())
    p1 = np.concatenate(p1s); p2 = np.concatenate(p2s)
    g1 = np.concatenate(g1s); g2 = np.concatenate(g2s)
    def report(y, yhat):
        return dict(
            MAE = float(mean_absolute_error(y, yhat)),
            RMSE= float(np.sqrt(mean_squared_error(y, yhat))),
            R2  = float(r2_score(y, yhat)),
        )
    return report(g1, p1), report(g2, p2)

if SHOW_TEST:
    dpeak_metrics, auc_metrics = eval_loader(teL)
    print("=== TEST (held‑out) ===")
    print(f"d_peak : MAE={dpeak_metrics['MAE']:.3f}  RMSE={dpeak_metrics['RMSE']:.3f}  R2={dpeak_metrics['R2']:.3f}")
    print(f"auc_0_2h: MAE={auc_metrics['MAE']:.3f}  RMSE={auc_metrics['RMSE']:.3f}  R2={auc_metrics['R2']:.3f}")
